In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
import numpy as np
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 1. Load MNIST
# -----------------------------
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False)

# Extract all data into tensors
all_images = []
all_labels = []

for images, labels in train_loader:
    all_images.append(images.view(images.size(0), -1))
    all_labels.append(labels)

X = torch.cat(all_images).numpy()  # shape: (60000, 784)
y = torch.cat(all_labels).numpy()  # shape: (60000,)

# -----------------------------
# Helper: map clusters to labels
# -----------------------------
def cluster_accuracy(y_true, y_cluster):
    label_map = {}
    for cluster_id in np.unique(y_cluster):
        idx = np.where(y_cluster == cluster_id)[0]
        majority_label = Counter(y_true[idx]).most_common(1)[0][0]
        label_map[cluster_id] = majority_label
    
    y_pred = np.vectorize(label_map.get)(y_cluster)
    return accuracy_score(y_true, y_pred)

# -----------------------------
# 2. Case A: Raw pixels + k-means
# -----------------------------
print("Running k-means on raw pixels...")

kmeans_raw = KMeans(n_clusters=10, n_init=10, random_state=42)
clusters_raw = kmeans_raw.fit_predict(X)

acc_raw = cluster_accuracy(y, clusters_raw)
print(f"Raw pixel clustering accuracy: {acc_raw:.4f}")

# -----------------------------
# 3. Case B: Neural network latent space
# -----------------------------
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        z = self.encoder(x)
        out = self.classifier(z)
        return out, z

model = SimpleNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train for a few epochs
print("Training neural network...")
for epoch in range(5):
    for images, labels in DataLoader(train_dataset, batch_size=512, shuffle=True):
        images = images.view(images.size(0), -1).to(device)
        labels = labels.to(device)

        outputs, _ = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# -----------------------------
# Extract latent features
# -----------------------------
print("Extracting latent features...")

latent_features = []

with torch.no_grad():
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)
        _, z = model(images)
        latent_features.append(z.cpu())

Z = torch.cat(latent_features).numpy()

# -----------------------------
# k-means in latent space
# -----------------------------
print("Running k-means on latent space...")

kmeans_latent = KMeans(n_clusters=10, n_init=10, random_state=42)
clusters_latent = kmeans_latent.fit_predict(Z)

acc_latent = cluster_accuracy(y, clusters_latent)
print(f"Latent space clustering accuracy: {acc_latent:.4f}")

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:01<00:00, 8.41MB/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 270kB/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 2.56MB/s]


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 601kB/s]


Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw

Running k-means on raw pixels...
Raw pixel clustering accuracy: 0.5908
Training neural network...
Extracting latent features...
Running k-means on latent space...
Latent space clustering accuracy: 0.9379


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
import numpy as np
from collections import Counter

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# Helper: Cluster accuracy
# -----------------------------
def cluster_accuracy(y_true, y_cluster):
    label_map = {}
    for cluster_id in np.unique(y_cluster):
        idx = np.where(y_cluster == cluster_id)[0]
        majority_label = Counter(y_true[idx]).most_common(1)[0][0]
        label_map[cluster_id] = majority_label
    
    y_pred = np.vectorize(label_map.get)(y_cluster)
    return accuracy_score(y_true, y_pred)

# -----------------------------
# Load MNIST
# -----------------------------
transform = transforms.ToTensor()

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=512, shuffle=False)

# -----------------------------
# Extract RAW pixel features
# -----------------------------
print("\nExtracting raw pixel features...")

X_train = []
y_train = []

for images, labels in train_loader:
    X_train.append(images.view(images.size(0), -1))
    y_train.append(labels)

X_train = torch.cat(X_train).numpy()
y_train = torch.cat(y_train).numpy()

X_test = []
y_test = []

for images, labels in test_loader:
    X_test.append(images.view(images.size(0), -1))
    y_test.append(labels)

X_test = torch.cat(X_test).numpy()
y_test = torch.cat(y_test).numpy()

# -----------------------------
# Case A: k-means on raw pixels
# -----------------------------
print("\nRunning k-means on raw pixels...")

kmeans_raw = KMeans(n_clusters=10, n_init=10, random_state=42)
kmeans_raw.fit(X_train)

clusters_test_raw = kmeans_raw.predict(X_test)
acc_raw = cluster_accuracy(y_test, clusters_test_raw)

print(f"Raw pixel TEST clustering accuracy: {acc_raw:.4f}")

# -----------------------------
# Define Neural Network
# -----------------------------
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        z = self.encoder(x)
        out = self.classifier(z)
        return out, z

model = SimpleNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -----------------------------
# Train Neural Network
# -----------------------------
print("\nTraining neural network...")

for epoch in range(5):
    model.train()
    total_loss = 0
    
    for images, labels in train_loader:
        images = images.view(images.size(0), -1).to(device)
        labels = labels.to(device)

        outputs, _ = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# -----------------------------
# Extract Latent Features
# -----------------------------
print("\nExtracting latent features...")

def extract_latent(loader):
    model.eval()
    features = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.view(images.size(0), -1).to(device)
            _, z = model(images)
            features.append(z.cpu())
    return torch.cat(features).numpy()

Z_train = extract_latent(train_loader)
Z_test  = extract_latent(test_loader)

# -----------------------------
# Case B: k-means in latent space
# -----------------------------
print("\nRunning k-means on latent space...")

kmeans_latent = KMeans(n_clusters=10, n_init=10, random_state=42)
kmeans_latent.fit(Z_train)

clusters_test_latent = kmeans_latent.predict(Z_test)
acc_latent = cluster_accuracy(y_test, clusters_test_latent)

print(f"Latent space TEST clustering accuracy: {acc_latent:.4f}")

Using device: cuda

Extracting raw pixel features...

Running k-means on raw pixels...
Raw pixel TEST clustering accuracy: 0.5944

Training neural network...
Epoch 1, Loss: 75.4076
Epoch 2, Loss: 29.1882
Epoch 3, Loss: 21.8924
Epoch 4, Loss: 17.1367
Epoch 5, Loss: 13.9515

Extracting latent features...

Running k-means on latent space...
Latent space TEST clustering accuracy: 0.9080
